In [9]:
import os, json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

# CONFIG: update to match your sheet
PROJECT_ID_COL = 'Project: Project Name'   # e.g., 'Project', 'Id'
DATE_COL       = 'Reporting Date'         # snapshot date column in template
PREDICT_DATE   = pd.Timestamp('2025-11-13')  # the single date to predict for (change as needed)

DATA_PATH   = r'Predict_template_ME_Billable_Customer_Projs.xlsx'
ARTIFACT_DIR = Path('model_artifacts')
MODEL_PATH   = ARTIFACT_DIR / 'gb_rag_status.pkl'
FEATURES_PATH= ARTIFACT_DIR / 'gb_expected_features.json'

target_cols = ['Overall Status', 'Revenue Status', 'Schedule Status', 'Client Blockage Status']

# Load prediction template
df = pd.read_excel(DATA_PATH)

# If template contains multiple snapshots per project, collapse to 1 row per project at/before PREDICT_DATE
if DATE_COL in df.columns:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
    df = (
        df[df[DATE_COL] <= PREDICT_DATE]
        .sort_values([PROJECT_ID_COL, DATE_COL])
        .groupby(PROJECT_ID_COL, as_index=False)
        .tail(1)
    )

# Load artifacts
model = joblib.load(MODEL_PATH)
with open(FEATURES_PATH, 'r', encoding='utf-8') as f:
    expected_features = json.load(f)

def preprocess_and_align(X_raw: pd.DataFrame, expected_features: list[str]) -> pd.DataFrame:
    X_proc = X_raw.copy()
    # Drop targets/IDs/dates to avoid leakage
    X_proc = X_proc.drop(columns=[c for c in target_cols if c in X_proc.columns], errors='ignore')
    X_proc = X_proc.drop(columns=[PROJECT_ID_COL, DATE_COL], errors='ignore')

    # Basic impute + one-hot
    num_cols = X_proc.select_dtypes(include='number').columns
    if len(num_cols) > 0:
        X_proc[num_cols] = X_proc[num_cols].fillna(X_proc[num_cols].median(numeric_only=True))
    X_proc = pd.get_dummies(X_proc, dummy_na=True)

    # Align to training schema
    for c in expected_features:
        if c not in X_proc.columns:
            X_proc[c] = 0
    extra = [c for c in X_proc.columns if c not in expected_features]
    if extra:
        X_proc = X_proc.drop(columns=extra)
    X_proc = X_proc[expected_features]
    return X_proc

# Build features and predict
X_aligned = preprocess_and_align(df, expected_features)
try:
    y_pred = model.predict(X_aligned)
except ValueError:
    y_pred = model.predict(X_aligned.to_numpy(dtype=np.float64))

pred_df = pd.DataFrame(y_pred, columns=target_cols, index=df.index)

out_df = df.copy()
for c in target_cols:
    out_df[f'Predicted - {c}'] = pred_df[c]

# Keep the same output path
OUT_PATH = ARTIFACT_DIR / 'predictions.xlsx'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
out_df.to_excel(OUT_PATH, index=False)
print(f'Predictions saved to: {OUT_PATH}')

C:\Users\kumarni\AppData\Local\Temp\1\ipykernel_30420\3164866552.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_proc[c] = 0
C:\Users\kumarni\AppData\Local\Temp\1\ipykernel_30420\3164866552.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_proc[c] = 0
C:\Users\kumarni\AppData\Local\Temp\1\ipykernel_30420\3164866552.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

Predictions saved to: model_artifacts\predictions.xlsx
